# <div style="text-align: center; direction: rtl; font-family: Vazir;"><h1 align="center" style="font-size: 24px; padding: 20px;">⚜️━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━⚜️<br>سوال دوم: تحلیل و مقایسه معماری‌های Encoder–Decoder و Decoder-Only در تسک Text-to-SQL <br>⚜️━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━⚜️</h1></div>


<p dir="rtl" style="text-align: right; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: Vazir;">
در این بخش از تمرین با مسئله Text-to-SQL آشنا می‌شوید؛ وظیفه‌ای که در آن مدل باید پرسش‌های زبان طبیعی را به کوئری SQL معتبر تبدیل کند. هدف تمرین، مقایسه عملی دو معماری مهم ترنسفورمری است: Encoder–Decoder‌ (مانند BART) و Decoder-only‌ (مانند GPT-2). پس از آماده‌سازی داده‌ها، ابتدا مدل BART و سپس مدل GPT-2 را برای این تسک آموزش می‌دهید و در نهایت عملکرد دو مدل را از نظر دقت، رفتار تولیدی و انواع خطاها تحلیل و مقایسه خواهید کرد.
</p>




## <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش اول: مجموعه‌داده و پیش‌پردازش(10 نمره)</div>


<div dir="rtl" style="text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
داده‌های Gretel Synthetic Text-to-SQL را از HuggingFace بارگذاری کرده و فایل‌های train/test را دریافت کنید. ساختار دیتاست را بررسی کرده، یک نمونه نمایش دهید و نقش ستون‌های اصلی از جمله question، schema و query را توضیح دهید. سپس داده‌ها را با استفاده از train_test_split به بخش‌های train و dev تقسیم کنید و ستون‌ها را به اسامی استاندارد (question، schema، query) تغییر دهید. برای کاهش زمان آموزش، روی هر بخش محدودیت اندازه تعیین کنید. در پایان، تحلیل کوتاهی از ماهیت پرسش‌ها، ساختار schemaها و چالش‌هایی که برای مدل ایجاد می‌کنند ارائه دهید.
</div>


<p dir='rtl' style="line-height: 2.0; text-align: right; font-family: Vazir; font-size: 16px; margin-top: 20px; color: white; background-color:rgb(0, 40, 30); padding: 30px; border-radius: 8px;">
🎯 <b>خروجی مورد انتظار:</b><br>
در پایان این بخش انتظار می‌رود دانشجو یک دیتافریم کاملا آماده برای مراحل بعدی در اختیار داشته باشد. خروجی مطلوب شامل بارگذاری صحیح فایل‌های train و test، نمایش ساختار کلی داده‌ها، تقسیم درست داده‌ها به بخش‌های train و dev، تغییر نام ستون‌ها به قالب استاندارد، و اعمال زیرنمونه‌گیری برای محدود کردن حجم داده‌ها است. علاوه بر این، باید یک نمونه واقعی از داده‌ها همراه با توضیح مختصر درباره ماهیت سؤال، ساختار schema و نمونه SQL نمایش داده شود تا مشخص شود داده‌ها به‌درستی پردازش شده و برای ورود به مراحل آموزش مدل آماده هستند.

</p>

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

dataset = load_dataset("gretelai/synthetic_text_to_sql")

train_raw = dataset["train"].to_pandas()
test_raw = dataset["test"].to_pandas()

print("Train shape (raw):", train_raw.shape)
print("Test shape  (raw):", test_raw.shape)
print("Raw columns:", train_raw.columns.tolist())

def prepare_split(df: pd.DataFrame, max_samples: int | None = None) -> pd.DataFrame:
    rename_map = {
        "sql_prompt": "question",
        "sql_context": "schema",
        "sql": "query"
    }

    df = df.rename(columns=rename_map)

    df = df[["question", "schema", "query"]]

    df = df.dropna(subset=["question", "schema", "query"])

    if max_samples is not None:
        df = df.iloc[:max_samples].reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df

train_tmp, dev_tmp = train_test_split(
    train_raw,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

MAX_TRAIN_SAMPLES = 5000
MAX_DEV_SAMPLES   = 1000
MAX_TEST_SAMPLES  = 2000

train_df = prepare_split(train_tmp, max_samples=MAX_TRAIN_SAMPLES)
dev_df   = prepare_split(dev_tmp,   max_samples=MAX_DEV_SAMPLES)
test_df  = prepare_split(test_raw,  max_samples=MAX_TEST_SAMPLES)

print("Train shape (final):", train_df.shape)
print("Dev shape   (final):", dev_df.shape)
print("Test shape  (final):", test_df.shape)

sample_idx = 0
print("\n=== Sample example from train_df ===")
print("Question:")
print(train_df.loc[sample_idx, "question"])
print("\nSchema:")
print(train_df.loc[sample_idx, "schema"])
print("\nQuery:")
print(train_df.loc[sample_idx, "query"])

data_splits = {
    "train": train_df,
    "dev": dev_df,
    "test": test_df
}

Train shape (raw): (100000, 11)
Test shape  (raw): (5851, 11)
Raw columns: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']
Train shape (final): (5000, 3)
Dev shape   (final): (1000, 3)
Test shape  (final): (2000, 3)

=== Sample example from train_df ===
Question:
What is the maximum number of lanes for highways in the European Union?

Schema:
CREATE TABLE highways (id INT, country VARCHAR(255), max_number_of_lanes INT); INSERT INTO highways (id, country, max_number_of_lanes) VALUES (1, 'Germany', 12), (2, 'France', 10), (3, 'Spain', 14), (4, 'Italy', 10), (5, 'Poland', 10), (6, 'Netherlands', 6), (7, 'Belgium', 8), (8, 'Greece', 6), (9, 'Portugal', 8), (10, 'Sweden', 8);

Query:
SELECT MAX(max_number_of_lanes) FROM highways WHERE country IN (SELECT name FROM countries WHERE continent = 'Europe');


<div dir='rtl' style='background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
✍️ <b>پاسخ تشریحی:</b><br>
در این بخش از دیتاست <code>gretelai/synthetic_text_to_sql</code> روی HuggingFace استفاده کردم. با <code>load_dataset</code> دو بخش <code>train</code> و <code>test</code> را گرفتم و به DataFrame پانداس تبدیل کردم. همان‌طور که در خروجی چاپ شد، دیتاست خام train حدود ۱۰۰هزار ردیف و ۱۱ ستون دارد و ستون‌های مهمش شامل اطلاعات دامنه، نوع تسک SQL، سطح پیچیدگی و در نهایت سه ستون اصلی <code>sql_prompt</code>، <code>sql_context</code> و <code>sql</code> هستند. این سه‌تا دقیقاً همان چیزهایی‌اند که برای Text-to-SQL لازم داریم.  

برای آماده‌سازی، یک تابع <code>prepare_split</code> نوشتم. داخل این تابع اول اسم ستون‌ها را به قالب استاندارد تمرین تغییر دادم: <code>sql_prompt → question</code>، <code>sql_context → schema</code>، <code>sql → query</code>.

بعد فقط همین سه ستون را نگه داشتم و با <code>dropna</code> ردیف‌هایی که یکی از این سه مقدار را نداشتند حذف کردم تا دیتاست تمیز شود. در ادامه برای این‌که آموزش سبک‌تر و قابل اجرا باشد، روی هر بخش یک <code>max_samples</code> گذاشتم؛ یعنی از train فقط ۵۰۰۰ نمونه، از dev هزار تا و از test دو هزار تا استفاده کردم. قبل از این مرحله هم خود train خام را با <code>train_test_split</code> به دو بخش <code>train_tmp</code> و <code>dev_tmp</code> با نسبت ۹۰/۱۰ و <code>random_state=42</code> تقسیم کردم تا هم train و هم dev تصادفی اما reproducible باشند.

در نهایت سه دیتافریم نهایی من این شد: <code>train_df</code> با سایز (۵۰۰۰ × ۳)، <code>dev_df</code> با سایز (۱۰۰۰ × ۳)، <code>test_df</code> با سایز (۲۰۰۰ × ۳).

برای این‌که مطمئن شوم همه‌چیز درست است، یک نمونه از <code>train_df</code> را چاپ کردم. در این نمونه: <br>
– ستون <b>question</b> یک سؤال به زبان طبیعی است: «حداکثر تعداد لاین‌های بزرگراه‌ها در اتحادیه اروپا چقدر است؟»
– ستون <b>schema</b> یک اسکیما به‌صورت متن SQL است که جدول <code>highways</code> را تعریف می‌کند و چند ردیف برای کشورهای مختلف (Germany, France, Spain, …) با تعداد لاین‌های متفاوت داخلش INSERT شده است.
– ستون <b>query</b> یک کوئری SQL هدف است که با <code>MAX</code> بیشترین تعداد لاین را از جدول <code>highways</code> برمی‌دارد و در شرط <code>WHERE</code> از یک زیرکوئری استفاده می‌کند تا فقط کشورهای قاره اروپا را انتخاب کند.

از نظر ماهیت کلی، پرسش‌ها معمولاً روی مفاهیم دنیای واقعی (سلامت، حمل‌ونقل، آموزش، محیط‌زیست و …) تعریف شده‌اند و اسکیماها به‌صورت <code>CREATE TABLE</code> + گاهی چند <code>INSERT</code> نوشته شده‌اند. این یعنی ورودی مدل شامل یک متن نسبتاً طولانی است که هم توضیح ساختار جدول‌ها را دارد، هم مقادیر نمونه را. چالش برای مدل این است که باید از روی سؤال تشخیص بدهد کدام جدول و کدام ستون‌ها مهم‌اند (مثلاً <code>max_number_of_lanes</code>، یا <code>country</code>)، نوع عمل مورد نیاز چیست (MAX، COUNT، AVG، GROUP BY، زیرکوئری و…) و در نهایت یک SQL سینتکس‌درست و منطقی تولید کند.

به طور خلاصه، در انتهای این بخش یک دیتافریم استاندارد و تمیز با ستون‌های <code>question</code>، <code>schema</code> و <code>query</code> برای هر سه split (train/dev/test) دارم که کاملاً آماده است تا مستقیم وارد مرحلهٔ ساخت Dataset هگینگ‌فیس و آموزش مدل‌های BART و GPT-2 شود.

</div>

## <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش دوم: آموزش مدل Encoder–Decoder (BART) (۱۵ نمره)</div>


<div dir="rtl" style="text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
<p style="line-height: 1.8; text-align: right;">
در این بخش از تمرین، وارد مرحله‌ی پیاده‌سازی مدل BART می‌شویم. پیش از شروع، لازم است به‌صورت مختصر و دقیق توضیح دهید که مدل BART چیست و چگونه کار می‌کند؛ به‌ویژه این‌که چرا معماری Encoder–Decoder آن برای وظایفی مانند Text-to-SQL مناسب است. انتظار می‌رود دانشجو به ساختار دو‌بخشی BART، نحوهٔ پردازش ورودی در انکودر، و تولید خروجی در دیکودر اشاره کند و نشان دهد که این معماری برای نگاشت دنباله‌ای از کلمات (سؤال + schema) به دنباله‌ای دیگر (SQL) کاملاً مناسب است.
پس از توضیح مدل، باید قالب ورودی مناسبی طراحی کنید که در آن سؤال و schema به‌صورت یک رشتهٔ یکپارچه در اختیار مدل قرار گیرند. سپس یک Dataset اختصاصی برای BART بسازید که وظایفی مانند tokenize کردن ورودی، ساخت attention mask‌ها و آماده‌سازی SQL به عنوان خروجی هدف مدل را انجام دهد. مدل را حداقل یک ایپاک روی ۲۰۰۰ نمونه آموزش دهید و loss نهایی، زمان اجرا و هایپرپارامترهای اصلی را گزارش کنید.
در مرحلهٔ ارزیابی، باید عملکرد مدل را روی dev و test با دو معیار Raw Exact Match و Normalized Exact Match بسنجید.
از شما انتظار می‌رود که هر دو معیار را توضیح دهید:
Raw EM چگونه مقایسهٔ مستقیم دو رشته را انجام می‌دهد، چرا ممکن است برای SQL ناکافی باشد، و Normalized EM چگونه با حذف تفاوت‌های سطحی (مثل فاصله‌ها، علامت‌ها و بزرگی یا کوچکی حروف) مقایسه‌ای عادلانه‌تر ارائه می‌دهد.
در پایان، پنج نمونهٔ تصادفی از dev انتخاب کنید و پرسش، خروجی مدل، SQL طلایی و نوع خطاها را تحلیل کنید.
<div dir="rtl" style="text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
نکته مهم:
برای پیاده‌سازی Raw EM می‌توانید از مقایسهٔ مستقیم رشته‌ها در پایتون استفاده کنید.
برای Normalized EM مجاز هستید از ابزارهایی مانند sqlparse برای فرمت‌بندی SQL و توابع سادهٔ پردازش متن جهت lowercase کردن و حذف فاصله‌های اضافی استفاده کنید، اما منطق مقایسه‌ی Normalized باید توسط خودتان نوشته شود.
</p>
</div>


<p dir='rtl' style="line-height: 2.0; text-align: right; font-family: Vazir; font-size: 16px; margin-top: 20px; color: white; background-color:rgb(0, 40, 30); padding: 30px; border-radius: 8px;">
🎯 <b>خروجی مورد انتظار:</b><br>
خروجی مورد انتظار این بخش، یک مدل BART آموزش‌دیده و ارزیابی‌شده است که بتواند از روی پرسش و ساختار پایگاه‌داده، کوئری SQL معتبر تولید کرده و معیارهای Raw EM و Normalized EM آن روی داده‌های dev و test گزارش شده باشد.
</p>

In [ ]:
import os
import time
import re
import random

import torch
from datasets import Dataset
from transformers import (
    BartTokenizerFast,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

os.environ["WANDB_DISABLED"] = "true"
!pip install -q sqlparse
import sqlparse

MODEL_NAME = "facebook/bart-base"

print("Loading tokenizer and model:", MODEL_NAME)
tokenizer = BartTokenizerFast.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
model.to(device)

BART_TRAIN_SIZE = 4000
BART_DEV_SIZE = 500
BART_TEST_SIZE = 1000

bart_train_df = train_df.iloc[:BART_TRAIN_SIZE].reset_index(drop=True)
bart_dev_df = dev_df.iloc[:BART_DEV_SIZE].reset_index(drop=True)
bart_test_df = test_df.iloc[:BART_TEST_SIZE].reset_index(drop=True)

print("BART train subset shape:", bart_train_df.shape)
print("BART dev subset shape:  ", bart_dev_df.shape)
print("BART test subset shape: ", bart_test_df.shape)

def build_input_text(question, schema):
    return "Question: {}\nSchema: {}".format(question, schema)

max_input_length = 512
max_target_length = 128

def preprocess_example(example):
    input_text = build_input_text(example["question"], example["schema"])
    model_inputs = tokenizer(
        input_text,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            example["query"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )["input_ids"]

    new_labels = []
    for token_id in labels:
        if token_id == tokenizer.pad_token_id:
            new_labels.append(-100)
        else:
            new_labels.append(token_id)

    model_inputs["labels"] = new_labels

    return model_inputs

train_dataset_hf = Dataset.from_pandas(bart_train_df)
dev_dataset_hf = Dataset.from_pandas(bart_dev_df)
test_dataset_hf = Dataset.from_pandas(bart_test_df)

print("Tokenizing datasets...")
train_tokenized = train_dataset_hf.map(
    preprocess_example,
    batched=False,
    remove_columns=train_dataset_hf.column_names,
)
dev_tokenized = dev_dataset_hf.map(
    preprocess_example,
    batched=False,
    remove_columns=dev_dataset_hf.column_names,
)
test_tokenized = test_dataset_hf.map(
    preprocess_example,
    batched=False,
    remove_columns=test_dataset_hf.column_names,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)


training_args = TrainingArguments(
    output_dir="bart_text2sql",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
)

print("Training hyperparameters:")
print(training_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=dev_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

start_time = time.time()
train_result = trainer.train()
end_time = time.time()

final_train_loss = train_result.training_loss
training_time_seconds = end_time - start_time

print("\n=== Training finished ===")
print("Final training loss:", final_train_loss)
print("Training time (seconds):", training_time_seconds)

def raw_exact_match(pred_sql, gold_sql):
    if pred_sql is None or gold_sql is None:
        return False
    return pred_sql.strip() == gold_sql.strip()


def normalize_sql(sql):
    if sql is None:
        return ""
    formatted = sqlparse.format(
        sql,
        keyword_case="lower",
        identifier_case="lower",
        strip_comments=True,
        reindent=False,
    )

    formatted = re.sub(r"\s+", " ", formatted)
    formatted = formatted.strip()

    return formatted


def normalized_exact_match(pred_sql, gold_sql):
    return normalize_sql(pred_sql) == normalize_sql(gold_sql)

def evaluate_bart_on_df(
    model,
    tokenizer,
    df,
    max_input_length_eval=512,
    max_new_tokens=64,
    num_beams=4,
):
    model.eval()

    raw_correct = 0
    norm_correct = 0
    n = len(df)

    for i in range(n):
        question = df.loc[i, "question"]
        schema = df.loc[i, "schema"]
        gold_sql = df.loc[i, "query"]

        input_text = build_input_text(question, schema)
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_length_eval,
        )

        for k in inputs:
            inputs[k] = inputs[k].to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask", None),
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                early_stopping=True,
            )

        pred_sql = tokenizer.decode(
            generated_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        if raw_exact_match(pred_sql, gold_sql):
            raw_correct += 1
        if normalized_exact_match(pred_sql, gold_sql):
            norm_correct += 1

    raw_em = float(raw_correct) / float(n)
    norm_em = float(norm_correct) / float(n)
    return raw_em, norm_em


print("\n=== Evaluating BART on dev subset ===")
dev_raw_em, dev_norm_em = evaluate_bart_on_df(
    model,
    tokenizer,
    bart_dev_df,
)

print("Dev Raw EM:        {:.4f}".format(dev_raw_em))
print("Dev Normalized EM: {:.4f}".format(dev_norm_em))

print("\n=== Evaluating BART on test subset ===")
test_raw_em, test_norm_em = evaluate_bart_on_df(
    model,
    tokenizer,
    bart_test_df,
)

print("Test Raw EM:        {:.4f}".format(test_raw_em))
print("Test Normalized EM: {:.4f}".format(test_norm_em))

print("\n=== Sample qualitative analysis on 5 dev examples ===")

k_samples = min(5, len(bart_dev_df))
random_indices = random.sample(range(len(bart_dev_df)), k=k_samples)

for idx in random_indices:
    question = bart_dev_df.loc[idx, "question"]
    schema = bart_dev_df.loc[idx, "schema"]
    gold_sql = bart_dev_df.loc[idx, "query"]

    input_text = build_input_text(question, schema)
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    )
    for k in inputs:
        inputs[k] = inputs[k].to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            max_new_tokens=64,
            num_beams=4,
            early_stopping=True,
        )

    pred_sql = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    raw_match = raw_exact_match(pred_sql, gold_sql)
    norm_match = normalized_exact_match(pred_sql, gold_sql)

    print("\n----------------------------------------")
    print("Example index:", idx)
    print("Question:")
    print(question)
    print("\nGold SQL:")
    print(gold_sql)
    print("\nPredicted SQL:")
    print(pred_sql)
    print("\nRaw EM match:        {}".format(raw_match))
    print("Normalized EM match: {}".format(norm_match))


Loading tokenizer and model: facebook/bart-base


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-2305227903.py:115: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training hyperparameters:
TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_

Step,Training Loss
50,2.091200
100,1.260200
150,1.139700
200,0.979600
250,0.910500
300,0.768800
350,0.732100
400,0.730400
450,0.717700
500,0.696400


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



=== Training finished ===
Final training loss: 0.8646321563720704
Training time (seconds): 1019.4946665763855

=== Evaluating BART on dev subset ===
Dev Raw EM:        0.1660
Dev Normalized EM: 0.1700

=== Evaluating BART on test subset ===
Test Raw EM:        0.1790
Test Normalized EM: 0.1800

=== Sample qualitative analysis on 5 dev examples ===

----------------------------------------
Example index: 327
Question:
What are the names and locations of all excavation sites that have yielded bronze age artifacts?

Gold SQL:
SELECT ExcavationSites.SiteName, ExcavationSites.Location FROM ExcavationSites INNER JOIN Artifacts ON ExcavationSites.SiteID = Artifacts.SiteID WHERE Artifacts.Age = 'Bronze Age';

Predicted SQL:
SELECT SiteName, Location FROM ExcavationSites WHERE Age = Bronze Age;

Raw EM match:        False
Normalized EM match: False

----------------------------------------
Example index: 57
Question:
What is the total number of public participation events in the justice sector

<div dir='rtl' style='background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
✍️ <b>پاسخ تشریحی:</b><br>

در این بخش از تمرین از مدل BART برای تسک Text-to-SQL استفاده کردم. به طور خلاصه، BART یک مدل Encoder–Decoder مبتنی بر ترنسفورمر است که ورودی را با یک انکودر دوجهته می‌خواند و خروجی را با یک دیکودر خودبازگشتی توکن به توکن تولید می‌کند. در اینجا ورودی ما «سؤال کاربر + ساختار پایگاه‌داده (schema)» است و خروجی یک کوئری SQL متنی است؛ یعنی دقیقاً یک مسأله sequence-to-sequence، که معماری BART برایش خیلی مناسب است: انکودر کل سؤال و schema را می‌فهمد و دیکودر بر اساس آن، دنباله‌ی SQL را مرحله‌به‌مرحله می‌سازد.

برای این‌که ورودی برای BART قابل فهم باشد، یک قالب متنی طراحی کردم به صورت: <br> <code>Question: ...<br>Schema: ...</code><br>
یعنی در هر مثال، سؤال و schema را به یک رشته‌ی مشترک تبدیل می‌کنم و به انکودر می‌دهم. خروجی هدف هم رشته‌ی SQL طلایی است. در تابع <code>preprocess_example</code> این کار را انجام دادم: متن ورودی را با <code>BartTokenizerFast</code> توکنایز کردم (با حداکثر طول ۵۱۲ توکن) و برای خروجی هم query را به توکن‌های هدف تبدیل کردم (حداکثر طول ۱۲۸). برای این‌که روی توکن‌های PAD خطا حساب نشود، آن‌ها را به -100 تبدیل کردم تا CrossEntropyLoss نادیده‌شان بگیرد. این پیش‌پردازش روی سه دیتاست train، dev و test اعمال شد و با Dataset و DataCollator مخصوص BART، برای Trainer آماده شد.

برای آموزش، از مدل پیش‌آموزش‌دیده‌ی <code>facebook/bart-base</code> استفاده کردم و روی ۴۰۰۰ نمونه‌ی train به مدت ۳ ایپاک با batch size=16 و learning rate=5e-5 آن را fine-tune کردم. خروجی آموزش نشان داد که loss به شکل منطقی کاهش پیدا کرده و در نهایت <b>training loss حدود ۰٫۸۶</b> شد. زمان آموزش تقریباً ۱۰۱۹ ثانیه (حدود ۱۷ دقیقه) بود.

برای ارزیابی، دو متریک Raw Exact Match و Normalized Exact Match را پیاده‌سازی کردم. در Raw EM فقط رشته‌ی SQL پیش‌بینی‌شده و SQL طلایی را به صورت مستقیم با هم مقایسه می‌کنم (بعد از strip). این روش خیلی سخت‌گیر است، چون کوچک‌ترین تفاوت در فاصله، حروف بزرگ/کوچک یا حتی ترتیب فاصله‌ها باعث می‌شود خروجی غلط حساب شود. به همین دلیل، برای SQL معمولاً کافی نیست. برای Normalized EM، قبل از مقایسه، هر دو SQL را با <code>sqlparse</code> و چند تابع ساده‌ی متنی نرمال‌سازی کردم: حروف را به lowercase تبدیل کردم، کامنت‌ها و فاصله‌های اضافی را حذف کردم و بین توکن‌ها یک فاصله‌ی استاندارد گذاشتم. بعد از این نرمال‌سازی، اگر دو رشته برابر بودند، Normalized EM را ۱ در نظر گرفتم. این کار کمک می‌کند تفاوت‌های سطحی نادیده گرفته شوند و تمرکز روی ساختار منطقی کوئری باشد.

نتایج روی dev و test به این صورت شد: <br>
Dev Raw EM ≈ 0.166 ، Dev Normalized EM ≈ 0.170
Test Raw EM ≈ 0.179 ، Test Normalized EM ≈ 0.180

یعنی مدل حدود ۱۷–۱۸٪ از کوئری‌ها را دقیقاً (یا بعد از نرمال‌سازی) درست تولید کرده است. فاصله‌ی کم بین Raw و Normalized EM نشان می‌دهد که بخش عمده‌ی خطاها ناشی از اختلاف‌های ظاهری نیست، بلکه بیشتر اختلاف‌های ساختاری و معنایی در خود SQL است.

برای تحلیل کیفی، پنج مثال تصادفی از dev انتخاب کردم و سؤال، SQL طلایی و خروجی مدل را نگاه کردم. در همه‌ی این مثال‌ها Raw EM و Normalized EM هر دو False بودند. چند الگوی خطا را می‌شود دید:

در یک سؤال درباره‌ی «سایت‌های حفاری با آثار عصر برنز»، SQL طلایی شامل join بین جداول ExcavationSites و Artifacts بود، ولی مدل کوئری ساده‌تری نوشت که فقط روی یک جدول فیلتر می‌کرد و حتی literal 'Bronze Age' را هم درست نقل نکرد. این نشان می‌دهد مدل اسکلت WHERE را می‌فهمد، ولی join و مبدأ دقیق ستون‌ها را درست درک نکرده است.

در مثالی دیگر درباره‌ی «تعداد رویدادهای مشارکت عمومی در سه سال اخیر»، SQL طلایی سه سال آخر را به صورت پویا با MAX(year) - 2 حساب می‌کرد، اما مدل یک بازه‌ی ثابت (مثلاً ۲۰۱۸ تا ۲۰۲۲) گذاشته بود. یعنی مفهوم «سه سال اخیر» را به‌درستی به یک بازه‌ی پویا نگذاشته و فقط به‌صورت حدودی یک بازه‌ی زمانی نوشته است.

در سؤال مربوط به «تغییر تعداد کتابخانه‌ها بین ۲۰۱۹ و ۲۰۲۰»، SQL طلایی از window function (LEAD) استفاده می‌کرد تا اختلاف دو سال پشت‌سرهم را در هر zone حساب کند، ولی مدل فقط SUM روی دو سال را نوشته بود و هیچ GROUP BY یا تابع window نداشت. در واقع مفهوم «تغییر» را به «جمع» تبدیل کرده بود.

در مثال نرخ چاقی در کالیفرنیا، SQL طلایی میانگین BMI را حساب می‌کرد، ولی مدل صرفاً <code>SELECT *</code> با WHERE روی State و Year تولید کرد، یعنی عملاً خروجی خام را می‌گیرد و شاخصی حساب نمی‌کند.

در مثال برندهای غیر وگان، مدل علاوه بر این‌که شرط‌ها (مثل Makeup و is_vegan=false و داشتن بیش از ۱۰ محصول) را رعایت نکرد، حتی علامت is_vegan را برعکس گذاشت و برندهای وگان را شمرد. این نشان می‌دهد که در ترکیب چند شرط و بخش HAVING مدل دچار سردرگمی می‌شود.

در مجموع، BART با این تنظیمات توانسته تا حدی ساختار کلی کوئری‌ها را یاد بگیرد و روی بخشی از مثال‌ها جواب دقیق بدهد، اما برای دیتاست Text-to-SQL پیچیده‌ای که استفاده کردم، هنوز EM نسبتاً پایین است و خطاها بیشتر مفهومی هستند. با این حال، از نظر معماری، رویکرد Encoder–Decoder BART برای نگاشت (سؤال + schema) به SQL کاملاً مناسب است و این نتایج یک نقطه‌ی شروع معقول برای بهبودهای بعدی (داده‌ی بیشتر، ایپاک بیشتر، محدودیت‌های نحوی و غیره) محسوب می‌شود.

</div>


## <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش سوم: آموزش مدل Decoder-only (GPT-2) (۱۵ نمره)</div>


<div dir="rtl" style="text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
<p style="line-height: 1.8; text-align: right;">
در این بخش تسک Text-to-SQL را با معماری GPT-2 پیاده‌سازی می‌کنید. ابتدا قالب ورودی مناسب برای مدل‌های decoder-only طراحی کنید، به‌طوری‌که پرسش و schema به‌صورت یک prefix مشخص (مانند question: … schema: … SQL:) به مدل داده شوند و مدل ادامه آن را به‌صورت SQL تولید کند. سپس یک Dataset مخصوص causal LM بسازید که full sequence (prefix + gold SQL + EOS) را تولید کرده و بخش prefix را در برچسب‌ها mask کند. مدل GPT-2 را یک epoch آموزش دهید و loss و زمان اجرا را گزارش کنید. هنگام ارزیابی، prefix را از خروجی حذف کنید و تنها SQL تولیدشده را بسنجید. معیارهای Raw EM و Normalized EM را روی dev و test گزارش کرده و ۵ نمونه از خروجی مدل را بررسی و تحلیل کنید.
</p>
</div>


<p dir='rtl' style="line-height: 2.0; text-align: right; font-family: Vazir; font-size: 16px; margin-top: 20px; color: white; background-color:rgb(0, 40, 30); padding: 30px; border-radius: 8px;">
🎯 <b>خروجی مورد انتظار:</b><br>
خروجی مورد انتظار این بخش، یک مدل GPT-2 آموزش‌دیده و ارزیابی‌شده است که با استفاده از قالب ورودی مبتنی بر prefix قادر به تولید SQL بوده و معیارهای Raw EM و Normalized EM آن روی مجموعه‌های dev و test به‌درستی محاسبه و گزارش شده باشند.
</p>

In [ ]:
import os
import time
import random
import re

import torch
from datasets import Dataset
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

!pip install -q sqlparse
import sqlparse

def raw_exact_match(pred_sql: str, gold_sql: str) -> bool:
    if pred_sql is None or gold_sql is None:
        return False
    return pred_sql.strip() == gold_sql.strip()

def normalize_sql(sql: str) -> str:
    if sql is None:
        return ""
    formatted = sqlparse.format(
        sql,
        keyword_case="lower",
        identifier_case="lower",
        strip_comments=True,
        reindent=False,
    )
    formatted = re.sub(r"\s+", " ", formatted)
    formatted = formatted.strip()
    return formatted

def normalized_exact_match(pred_sql: str, gold_sql: str) -> bool:
    return normalize_sql(pred_sql) == normalize_sql(gold_sql)

os.environ["WANDB_DISABLED"] = "true"

GPT2_MODEL_NAME = "gpt2"
print("Loading tokenizer and model:", GPT2_MODEL_NAME)
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained(GPT2_MODEL_NAME)
gpt2_model = GPT2LMHeadModel.from_pretrained(GPT2_MODEL_NAME)

if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
    gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
gpt2_model.to(device)

GPT2_TRAIN_SIZE = 4000
GPT2_DEV_SIZE   = 500
GPT2_TEST_SIZE  = 1000

gpt2_train_df = train_df.iloc[:GPT2_TRAIN_SIZE].reset_index(drop=True)
gpt2_dev_df   = dev_df.iloc[:GPT2_DEV_SIZE].reset_index(drop=True)
gpt2_test_df  = test_df.iloc[:GPT2_TEST_SIZE].reset_index(drop=True)

print("GPT-2 train subset shape:", gpt2_train_df.shape)
print("GPT-2 dev subset shape:  ", gpt2_dev_df.shape)
print("GPT-2 test subset shape: ", gpt2_test_df.shape)

def build_gpt2_prefix(question: str, schema: str) -> str:
    return f"question: {question}\nschema: {schema}\nSQL:"

max_seq_length_gpt2 = 256

def preprocess_gpt2_example(example):
    question = example["question"]
    schema   = example["schema"]
    gold_sql = example["query"]

    prefix_text = build_gpt2_prefix(question, schema)
    full_text   = prefix_text + " " + gold_sql + gpt2_tokenizer.eos_token

    encoded = gpt2_tokenizer(
        full_text,
        max_length=max_seq_length_gpt2,
        truncation=True,
        padding="max_length",
    )
    input_ids = encoded["input_ids"]

    prefix_ids = gpt2_tokenizer(
        prefix_text,
        max_length=max_seq_length_gpt2,
        truncation=True,
    )["input_ids"]

    labels = list(input_ids)

    prefix_len = len(prefix_ids)
    for i in range(prefix_len):
        if i < len(labels):
            labels[i] = -100

    labels = [
        (-100 if token_id == gpt2_tokenizer.pad_token_id else token_id)
        for token_id in labels
    ]

    encoded["labels"] = labels
    return encoded

gpt2_train_dataset_hf = Dataset.from_pandas(gpt2_train_df)
gpt2_dev_dataset_hf   = Dataset.from_pandas(gpt2_dev_df)
gpt2_test_dataset_hf  = Dataset.from_pandas(gpt2_test_df)

print("Tokenizing GPT-2 datasets...")
gpt2_train_tokenized = gpt2_train_dataset_hf.map(
    preprocess_gpt2_example,
    batched=False,
    remove_columns=gpt2_train_dataset_hf.column_names,
)
gpt2_dev_tokenized = gpt2_dev_dataset_hf.map(
    preprocess_gpt2_example,
    batched=False,
    remove_columns=gpt2_dev_dataset_hf.column_names,
)
gpt2_test_tokenized = gpt2_test_dataset_hf.map(
    preprocess_gpt2_example,
    batched=False,
    remove_columns=gpt2_test_dataset_hf.column_names,
)

gpt2_data_collator = DataCollatorForLanguageModeling(
    tokenizer=gpt2_tokenizer,
    mlm=False,
)

gpt2_training_args = TrainingArguments(
    output_dir="gpt2_text2sql",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
)

print("GPT-2 training hyperparameters:")
print(gpt2_training_args)

gpt2_trainer = Trainer(
    model=gpt2_model,
    args=gpt2_training_args,
    train_dataset=gpt2_train_tokenized,
    eval_dataset=gpt2_dev_tokenized,
    data_collator=gpt2_data_collator,
    tokenizer=gpt2_tokenizer,
)

gpt2_start_time = time.time()
gpt2_train_result = gpt2_trainer.train()
gpt2_end_time = time.time()

gpt2_final_train_loss = gpt2_train_result.training_loss
gpt2_training_time_seconds = gpt2_end_time - gpt2_start_time

print("\n=== GPT-2 training finished ===")
print("Final GPT-2 training loss:", gpt2_final_train_loss)
print("GPT-2 training time (seconds):", gpt2_training_time_seconds)

def evaluate_gpt2_on_df(
    model,
    tokenizer,
    df,
    max_prompt_length=256,
    max_new_tokens=64,
    num_beams=4,
):
    model.eval()
    raw_correct = 0
    norm_correct = 0
    n = len(df)

    for i in range(n):
        question = df.loc[i, "question"]
        schema   = df.loc[i, "schema"]
        gold_sql = df.loc[i, "query"]

        prefix_text = build_gpt2_prefix(question, schema)

        inputs = tokenizer(
            prefix_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_prompt_length,
        )

        input_ids     = inputs["input_ids"].to(model.device)
        attention_mask = inputs["attention_mask"].to(model.device)

        prefix_len = input_ids.shape[1]

        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                early_stopping=True,
            )

        full_sequence_ids = generated_ids[0]
        sql_ids = full_sequence_ids[prefix_len:]

        pred_sql = tokenizer.decode(
            sql_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        if raw_exact_match(pred_sql, gold_sql):
            raw_correct += 1
        if normalized_exact_match(pred_sql, gold_sql):
            norm_correct += 1

    raw_em  = float(raw_correct) / float(n)
    norm_em = float(norm_correct) / float(n)
    return raw_em, norm_em

print("\n=== Evaluating GPT-2 on dev subset ===")
gpt2_dev_raw_em, gpt2_dev_norm_em = evaluate_gpt2_on_df(
    gpt2_model,
    gpt2_tokenizer,
    gpt2_dev_df,
)
print("GPT-2 Dev Raw EM:        {:.4f}".format(gpt2_dev_raw_em))
print("GPT-2 Dev Normalized EM: {:.4f}".format(gpt2_dev_norm_em))

print("\n=== Evaluating GPT-2 on test subset ===")
gpt2_test_raw_em, gpt2_test_norm_em = evaluate_gpt2_on_df(
    gpt2_model,
    gpt2_tokenizer,
    gpt2_test_df,
)
print("GPT-2 Test Raw EM:        {:.4f}".format(gpt2_test_raw_em))
print("GPT-2 Test Normalized EM: {:.4f}".format(gpt2_test_norm_em))

print("\n=== GPT-2 sample qualitative analysis on 5 dev examples ===")
k_samples = min(5, len(gpt2_dev_df))
gpt2_random_indices = random.sample(range(len(gpt2_dev_df)), k=k_samples)

for idx in gpt2_random_indices:
    question = gpt2_dev_df.loc[idx, "question"]
    schema   = gpt2_dev_df.loc[idx, "schema"]
    gold_sql = gpt2_dev_df.loc[idx, "query"]

    prefix_text = build_gpt2_prefix(question, schema)

    inputs = gpt2_tokenizer(
        prefix_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_seq_length_gpt2,
    )
    input_ids      = inputs["input_ids"].to(gpt2_model.device)
    attention_mask = inputs["attention_mask"].to(gpt2_model.device)

    prefix_len = input_ids.shape[1]

    with torch.no_grad():
        generated_ids = gpt2_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            num_beams=4,
            early_stopping=True,
        )

    full_sequence_ids = generated_ids[0]
    sql_ids = full_sequence_ids[prefix_len:]

    pred_sql = gpt2_tokenizer.decode(
        sql_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    raw_match  = raw_exact_match(pred_sql, gold_sql)
    norm_match = normalized_exact_match(pred_sql, gold_sql)

    print("\n----------------------------------------")
    print("Example index:", idx)
    print("Question:")
    print(question)
    print("\nGold SQL:")
    print(gold_sql)
    print("\nPredicted SQL (GPT-2):")
    print(pred_sql)
    print("\nRaw EM match:        {}".format(raw_match))
    print("Normalized EM match: {}".format(norm_match))


Loading tokenizer and model: gpt2
Using device: cuda
GPT-2 train subset shape: (4000, 3)
GPT-2 dev subset shape:   (500, 3)
GPT-2 test subset shape:  (1000, 3)
Tokenizing GPT-2 datasets...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-4263910240.py:149: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  gpt2_trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


GPT-2 training hyperparameters:
TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,

Step,Training Loss
50,1.502600
100,1.159600
150,1.088800
200,1.050300
250,0.997500
300,0.936100
350,0.910200
400,0.905900
450,0.899700
500,0.888700



=== GPT-2 training finished ===
Final GPT-2 training loss: 0.9698961639404297
GPT-2 training time (seconds): 893.4913597106934

=== Evaluating GPT-2 on dev subset ===
GPT-2 Dev Raw EM:        0.0000
GPT-2 Dev Normalized EM: 0.0000

=== Evaluating GPT-2 on test subset ===
GPT-2 Test Raw EM:        0.0000
GPT-2 Test Normalized EM: 0.0000

=== GPT-2 sample qualitative analysis on 5 dev examples ===

----------------------------------------
Example index: 327
Question:
What are the names and locations of all excavation sites that have yielded bronze age artifacts?

Gold SQL:
SELECT ExcavationSites.SiteName, ExcavationSites.Location FROM ExcavationSites INNER JOIN Artifacts ON ExcavationSites.SiteID = Artifacts.SiteID WHERE Artifacts.Age = 'Bronze Age';

Predicted SQL (GPT-2):
 SELECT SiteName, Location FROM ExcavationSites WHERE Artifacts.SiteID = (SELECT SiteName FROM Artifacts WHERE Artifacts.Location = 'Bronze Age') AND Artifacts.Age > (SELECT SiteName FROM Artifacts WHERE Artifacts.Lo

<div dir='rtl' style='background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
✍️ <b>پاسخ تشریحی:</b><br>

در این بخش تسک Text-to-SQL را با معماری <b>Decoder-only</b> یعنی GPT-2 پیاده‌سازی کردم. تفاوت اصلی GPT-2 با BART این است که دیگر انکودر جداگانه نداریم؛ خود دیکودر (Language Model) کل ورودی را به صورت یک دنباله‌ی واحد می‌گیرد و به صورت <b>خودبازگشتی</b> (توکن به توکن از چپ به راست) ادامه‌ی آن را تولید می‌کند. برای همین، سؤال و schema را به‌صورت یک <b>prefix متنی</b> به مدل دادم و از آن خواستم ادامه‌ی این رشته را به شکل SQL تولید کند.

قالب ورودی را به صورت زیر طراحی کردم:

<code>question: <سؤال><br>schema: <اسکیما><br>SQL:</code>


برای هر نمونه، این prefix را ساختم و بعد <b>SQL طلایی + توکن EOS</b> را به آن چسباندم تا دنباله‌ی کامل به شکل <code>prefix + gold_sql + <EOS></code>
به دست بیاید. این دنباله را با <code>GPT2TokenizerFast</code> توکنایز کردم (با حداکثر طول ۲۵۶ توکن). برای ساخت دیتاست causal LM، ورودی مدل همان <code>input_ids</code> است و برای <code>labels</code> دقیقاً همین دنباله را قرار دادم، فقط بخش prefix را <b>mask</b> کردم (قرار دادن مقدار −100) تا در محاسبه‌ی loss نادیده گرفته شود. همچنین توکن‌های PAD هم در لیبل‌ها −100 شدند. به این ترتیب، مدل فقط روی قسمت SQL آموزش می‌بیند، در حالی که سؤال و schema فقط به عنوان زمینه‌ی شرطی‌کننده استفاده می‌شوند.

روی ۴۰۰۰ نمونه‌ی train، ۵۰۰ نمونه‌ی dev و ۱۰۰۰ نمونه‌ی test کار کردم. مدل GPT-2 پایه را با batch size برابر ۱۶، سه epoch و learning rate=5e-5 آموزش دادم. لاگ‌های آموزش نشان می‌دهد که loss به‌صورت یکنواخت پایین می‌آید (مثلاً از حدود ۱٫۵ در stepهای اولیه تا حدود ۰٫۸–۰٫۹ در انتها) و در نهایت <b>training loss ≈ 0.97</b> به دست آمد. زمان کل آموزش هم حدود <b>۸۹۳ ثانیه</b> بود که با توجه به اندازه‌ی مدل و داده منطقی است.

در مرحله‌ی ارزیابی، همان prefix قبلی را به مدل دادم، آن را توکنایز کردم و به GPT-2 پاس دادم. با <code>model.generate</code> و beam search، دنباله‌ی کامل خروجی را گرفتم، سپس با توجه به طول prefix، بخش مربوط به SQL را بریدم (یعنی از اندیس <code>prefix_len</code> تا انتهای sequence). این رشته‌ی SQL پیش‌بینی‌شده را با دو تابع <code>raw_exact_match</code> و <code>normalized_exact_match</code> (که قبلاً هم برای BART نوشته بودم) با SQL طلایی مقایسه کردم. Raw EM بر اساس مقایسه‌ی مستقیم رشته‌ها است و Normalized EM بعد از نرمال‌سازی با sqlparse (lowercase، حذف فاصله‌های اضافی و غیره) این مقایسه را انجام می‌دهد.

نتایج روی dev و test به شکل زیر بود:
GPT-2 Dev Raw EM = 0.0 ، Dev Normalized EM = 0.0
GPT-2 Test Raw EM = 0.0 ، Test Normalized EM = 0.0
یعنی در این تنظیمات، مدل حتی یک کوئری را هم دقیقاً مطابق SQL طلایی تولید نکرد؛ نه به صورت خام و نه بعد از نرمال‌سازی. این نتیجه با این‌که در نگاه اول ناامیدکننده است، ولی با توجه به سختی تسک و مدل انتخابی قابل‌توضیح است: GPT-2 فقط یک LM متن آزاد است و هیچ تضمین نحوی برای SQL ندارد، اندازه‌ی دیتاست برای چنین تسکی کم است، و معیار EM هم بسیار سخت‌گیرانه است (کوچک‌ترین اختلاف در joinها، توابع تجمیعی، شرط‌ها و حتی ترتیب شرط‌ها باعث می‌شود خروجی غلط حساب شود).

در تحلیل کیفی ۵ نمونه‌ی dev چند الگوی جالب دیده می‌شود. برای سؤال مربوط به «سایت‌های حفاری با آثار عصر برنز»، مدل یک کوئری تولید کرد که اسامی ستون‌ها و جدول ExcavationSites را به‌درستی استفاده می‌کند، ولی join با جدول Artifacts و شرط صحیح را اشتباه و پیچیده تولید کرده و چند زیرکوئری غیرلازم ساخته است. در مثال «تعداد رویدادهای مشارکت عمومی در سه سال اخیر»، مدل فقط <code>COUNT(*)</code> روی sector=‘justice’ نوشت و کل مفهوم «سه سال اخیر» و بازه‌ی زمانی پویا را حذف کرد. در سؤال «تغییر تعداد کتابخانه‌ها بین ۲۰۱۹ و ۲۰۲۰»، به جای محاسبه‌ی اختلاف با window function، صرفاً یک <code>GROUP BY</code> ساده روی یک سال نوشت. در مثال نرخ چاقی، مثل BART، فقط <code>SELECT *</code> با فیلتر California و Year تولید کرد و اصلاً BMI و تابع AVG را محاسبه نکرد. و در مثال برندهای غیر وگان، شرط is_vegan را برعکس گذاشت و برندهای وگان را شمرد.

به طور کلی، می‌توان گفت GPT-2 با این تنظیمات بیشتر یاد گرفته است که <b>اسکلت ظاهری</b> کوئری و نام جداول/ستون‌ها را از روی schema و سؤال حدس بزند، اما در منطق دقیق SQL، مخصوصاً joinها، HAVING، window function و شرط‌های ترکیبی ضعیف عمل می‌کند. از طرف دیگر، چون تخمین را به صورت متن آزاد انجام می‌دهد و هیچ constraint صوری روی نحو SQL ندارد، حتی خطاهای کوچک هم باعث می‌شود EM صفر شود. با این وجود، این بخش نشان می‌دهد که معماری Decoder-only با یک prefix مناسب می‌تواند تا حدی رفتار Text-to-SQL را یاد بگیرد، ولی برای رسیدن به دقت بالا احتمالاً به داده‌ی بیشتر، تنظیمات قوی‌تر (مثلاً مدل بزرگ‌تر یا instruction tuning) و یا تلفیق با قیدهای نحوی/ساختاری نیاز داریم.

</div>

### <div style="text-align: center; direction: rtl; font-family: Vazir;">بخش چهارم: مقایسه و تحلیل نهایی (۱۰ نمره)</div>


<div dir="rtl" style="text-align: right; padding:10px; background-color:#6B7280;  border-radius: 12px; border: 2px solid rgb(2, 34, 22); font-family: Vazir;">
<p style="line-height: 1.8; text-align: right;">
در این بخش باید عملکرد BART و GPT-2 را از جنبه‌های مختلف مقایسه کنید. در یک تحلیل جامع توضیح دهید که هر مدل چه نوع خطاهایی دارد، در استفاده از schema چگونه عمل می‌کند و دلیل تفاوت رفتار این دو معماری چیست. در پایان، جمع‌بندی مختصری ارائه دهید و بیان کنید در شرایط مختلف کدام رویکرد مناسب‌تر است و چه بهبودهایی می‌توان برای هر مدل پیشنهاد کرد.</p>
</div>


<p dir='rtl' style="line-height: 2.0; text-align: right; font-family: Vazir; font-size: 16px; margin-top: 20px; color: white; background-color:rgb(0, 40, 30); padding: 30px; border-radius: 8px;">
🎯 <b>خروجی مورد انتظار:</b><br>
خروجی مورد انتظار این بخش، یک تحلیل جامع و مقایسه‌محور است که در آن عملکرد، خطاها، نقاط قوت و ضعف دو مدل BART و GPT-2 بر اساس معیارهای ارزیابی، نمونه‌های تولیدی و تفاوت‌های معماری به‌طور دقیق بررسی و جمع‌بندی شده باشد.
</p>

<div dir='rtl' style='background:#fffbe6; font-family: Vazir; border:1px dashed #f0ad4e; padding:12px; border-radius:8px; color:#111'>
✍️ <b>پاسخ تشریحی:</b><br>
در این بخش دو مدل BART (Encoder–Decoder) و GPT-2 (Decoder-only) را هم از نظر عددی و هم از نظر نوع خطاها با هم مقایسه میکنیم. از نظر معیارها، BART روی dev و test به Raw EM و Normalized EM حدود ۰.۱۷–۰.۱۸ رسید، در حالی که GPT-2 عملاً روی هر دو مجموعه ۰ گرفت. این یعنی با این تنظیمات، BART حداقل در تعداد قابل توجهی از نمونه‌ها کوئری دقیقاً مطابق SQL طلایی تولید کرده، ولی GPT-2 حتی یک کوئری که دقیقاً (یا بعد از نرمال‌سازی) برابر طلایی باشد، نداشته است؛ هرچند خروجی‌های GPT-2 کاملاً بی‌ربط هم نیستند و معمولاً از نظر معنایی «نیمه‌درست» هستند.

از نظر نوع خطاها، BART معمولاً اسکیمای ورودی را بهتر می‌بیند و اسم جدول‌ها و ستون‌ها را درست برمی‌دارد، ولی در منطق کوئری ساده‌سازی می‌کند. مثلاً در مثال حفاری‌ها، به‌جای join صحیح بین <code>ExcavationSites</code> و <code>Artifacts</code>، فقط روی یک جدول شرط می‌گذارد یا شرط سن را روی ستون اشتباه می‌گذارد. در مثال کتابخانه‌ها و زون‌ها هم seen بود که به‌جای استفاده از توابع پنجره‌ای مثل <code>LEAD</code>، به سمت جمع‌زدن ساده یا شرط بین دو سال می‌رود. یعنی BART اغلب «اسکلت» SQL را درست می‌سازد (SELECT، FROM، WHERE, GROUP BY)، اما در بخش‌های ظریف مثل joinهای دقیق، شرایط زمانی متغیر و توابع تحلیلی دچار خطا می‌شود.

GPT-2 رفتار متفاوتی دارد. این مدل معمولاً به سراغ الگوهای خیلی عمومی و پرتکرار می‌رود، مثل <code>SELECT * FROM ... WHERE ...</code> یا <code>SELECT COUNT(*) FROM ... WHERE ...</code> و بخش‌های پیچیده‌تر سؤال را عملاً نادیده می‌گیرد. در مثال نرخ چاقی، فقط یک کوئری ساده با شرط State و Year تولید می‌کند و کلاً محاسبه‌ی BMI و میانگین را کنار می‌گذارد. در مثال برندهای غیر وگان، به‌جای شرط <code>is_vegan = false</code> و داشتن <code>HAVING COUNT(...) > 10</code>، فقط یک شمارش ساده روی <code>is_vegan = true</code> می‌نویسد که حتی از نظر منطقی برعکس سؤال است. علاوه بر این، در خروجی‌های GPT-2 تکرار الگو هم دیده می‌شود (مثلاً چند بار پشت‌سرهم «SQL: SELECT ...»)، که ناشی از ماهیت مدل زبانی و نبود شرط توقف معنایی است. به همین خاطر، حتی وقتی قسمتی از کوئری درست است، به خاطر اضافه‌گویی و ساختار غیرسازگار، دقیقاً با SQL طلایی match نمی‌شود و EM صفر می‌ماند.

اگر این تفاوت را به معماری ربط بدهیم، BART یک مدل Encoder–Decoder است: در انکودر، کل دنباله‌ی سؤال و schema را به صورت یک نمایش متنی فشرده می‌گیرد و دیکودر در هر گام با cross-attention دوباره به اسکیمای ورودی نگاه می‌کند. برای تسک‌هایی مثل Text-to-SQL که «نگاشت دقیق از ورودی به خروجی» می‌خواهند، این ساختار طبیعی‌تر است، چون دیکودر دائماً می‌تواند برگردد و ببیند کدام جدول/ستون در schema وجود دارد. در مقابل، GPT-2 فقط یک دیکودر causal است؛ یعنی سؤال و schema را به‌صورت prefix در همان دنباله‌ی ورودی می‌گیرد و بعد از آن فقط ادامه می‌دهد. مدل از نظر تئوری کل prefix را در حافظه دارد، اما در عمل وقتی prefix طولانی و schema پیچیده است، تمرکز روی جزئیات سخت‌تر می‌شود و مدل بیشتر به سمت الگوهای زبانی کلی (نه استنتاج دقیق مبتنی بر schema) می‌رود. به همین دلیل است که BART علی‌رغم این‌که هنوز فاصله‌ی زیادی با دقت ایده‌آل دارد، از GPT-2 در EM خیلی جلوتر است.

در جمع‌بندی، برای سناریوهایی مثل Text-to-SQL که نیاز به توجه دقیق به ساختار پایگاه‌داده و تولید خروجی ساخت‌یافته است، معماری‌های Encoder–Decoder مثل BART مناسب‌تر هستند. GPT-2 بیشتر شبیه یک «تکمیل‌کننده‌ی متن» رفتار می‌کند تا یک «ترجمه‌کننده‌ی دقیق سؤال به SQL»، مگر این‌که داده‌ی خیلی بزرگ‌تر، تنظیمات خاص‌تر و decoding کنترل‌شده‌تری (مثلاً grammar-based decoding) برایش در نظر بگیریم. برای بهبود BART می‌توان از تکنیک‌هایی مثل schema linking صریح، محدود کردن فضای جست‌وجو با گرامر SQL و آموزش طولانی‌تر استفاده کرد. برای GPT-2 هم می‌شود prefix را واضح‌تر ساخت، max length را بهتر تنظیم کرد، از محدودیت n-gram تکراری و post-processing برای بریدن خروجی بعد از اولین کوئری کامل استفاده کرد، اما در نهایت برای این نوع تسک، ذاتاً یک قدم عقب‌تر از مدل‌های Encoder–Decoder باقی می‌ماند.

</div>